# ESM Peptide Engineering + Structure Prediction Pipeline

This notebook implements a full computational workflow for peptide mutation analysis, stability scoring, and structural prediction using ESM models.

# Step One: Pipeline Overview

Step One: Define input peptide sequence provided by the user.  
Step Two: Generate all possible single-point mutations of the sequence.  
Step Three: Encode sequences using ESM protein language model embeddings.  
Step Four: Compute stability score based on embedding similarity.  
Step Five: Rank all generated variants.  
Step Six: Predict 3D structure for top-ranked variants.  
Step Seven: Export results as CSV file.

In [ ]:
!pip install -q torch fair-esm transformers biopython pandas numpy scipy scikit-learn matplotlib

# Step Two: Import Required Libraries

In this step we import all necessary libraries for sequence analysis, embedding generation, and numerical computation.

In [ ]:
import torch
import esm
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine
from transformers import AutoTokenizer, EsmForProteinFolding

# Step Three: Load Pretrained Models

We load the pretrained ESM model for embeddings and the ESMFold model for structure prediction.

In [ ]:
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()

# Step Four: Input Peptide Sequence (Editable by User)

In this step, the user provides the peptide sequence for analysis.

### Default mode (recommended for Run All):
If you simply run the notebook without changes, a high-quality biologically realistic peptide sequence will be used automatically.

### Custom mode:
You are free to replace the sequence below with your own peptide to test different hypotheses.

---

### Default reference sequence (high-quality protein fragment):
MKWVTFISLLFLFSSAYSRGVFRRDTHKSEIAHRFKDLGE

---

### Notes for advanced users:
- You may modify this cell to analyze any peptide of interest.
- Keep amino acids within the standard 20-letter alphabet for valid results.

In [ ]:
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

default_sequence = "MKWVTFISLLFLFSSAYSRGVFRRDTHKSEIAHRFKDLGE"

# ❌ NO INPUT AT ALL (for VSCode stability)
sequence = default_sequence

# validation
invalid = [aa for aa in sequence if aa not in VALID_AA]

if invalid:
    raise ValueError(f"Invalid amino acids: {set(invalid)}")

print("Sequence loaded safely (AUTO MODE)")
print("Length:", len(sequence))

sequence

# Step Five: Embedding Function

This function converts amino acid sequences into numerical representations using the ESM model.

In [ ]:
import torch

def embed(seq):

    inputs = tokenizer(
        seq,
        return_tensors="pt",
        truncation=True
    )

    with torch.no_grad():
        output = model(**inputs)

    embedding = output.last_hidden_state.mean(dim=1)

    return embedding.squeeze().cpu().numpy()


print("Embedding engine ready")

# Step Six: Mutation Landscape Generation (Core Engineering Module)

This step generates a full single-point mutation landscape for the input peptide.

### What this step does:
- Systematically mutates each residue
- Generates all possible single amino acid substitutions
- Produces a full mutational landscape for downstream analysis

### Advanced insight:
This module forms the foundation of:
- stability engineering
- hotspot detection
- evolutionary constraint analysis

---

### Customization note:
Advanced users may modify this cell to:
- restrict mutation space
- focus on specific residues
- apply biochemical filters

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"

def generate_mutants(seq):

    library = []

    for i in range(len(seq)):

        wt = seq[i]

        for aa in AA:

            if aa != wt:

                mutated = list(seq)
                mutated[i] = aa
                mutated = "".join(mutated)

                # IMPORTANT: ثابت و استاندارد
                library.append({
                    "position": i,
                    "wt": wt,
                    "mut": aa,
                    "sequence": mutated
                })

    return library


variants = generate_mutants(sequence)

print("Mutation library size:", len(variants))

# Step Seven: Stability Scoring

We compute a stability proxy score based on embedding similarity and aromatic residue content.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "facebook/esm2_t33_650M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

print("ESM model loaded successfully")

# Step Eight: Run Full Pipeline

We compute embeddings, generate mutants, and calculate scores for all variants.

In [ ]:
import numpy as np


def score(wt_emb, mut_emb, seq):

    similarity = np.dot(wt_emb, mut_emb)

    similarity /= (
        np.linalg.norm(wt_emb)
        *
        np.linalg.norm(mut_emb)
    )

    length_bonus = len(seq) / 100

    final_score = (
        0.9 * similarity
        +
        0.1 * length_bonus
    )

    return float(final_score)


print("Scoring engine ready")

In [ ]:
wt_emb = embed(sequence)

variants = generate_mutants(sequence)

MAX_VARIANTS = 20
variants = variants[:MAX_VARIANTS]

rows = []

for i, v in enumerate(variants):

    print(f"Processing {i+1}/{len(variants)}")

    mut_emb = embed(v["sequence"])

    s = score(
        wt_emb,
        mut_emb,
        v["sequence"]
    )

    rows.append([
        v["position"],
        v["wt"],
        v["mut"],
        v["sequence"],
        s
    ])

print("Analysis complete")

# Step Nine: Ranking Results

All variants are ranked based on their computed stability score.

In [ ]:
df = pd.DataFrame(rows, columns=["position", "wt", "mut", "sequence", "score"])
df = df.sort_values("score", ascending=False)

df.head(10)

# Step Ten: Structure Prediction

Top-ranked variants are passed to a structure prediction model (ESMFold) for 3D structure estimation.

In [ ]:
import torch
import gc
from transformers import AutoTokenizer, EsmForProteinFolding

print("Loading ESMFold...")

VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

if "df" not in globals():
    raise ValueError("df not found — run previous steps first.")

tokenizer = AutoTokenizer.from_pretrained(
    "facebook/esmfold_v1"
)

structure_model = EsmForProteinFolding.from_pretrained(
    "facebook/esmfold_v1"
)

if torch.cuda.is_available():
    structure_model = structure_model.cuda()

structure_model.eval()

print("Model ready")

top = df.head(2).reset_index(drop=True)

structures = []

for i, row in top.iterrows():

    raw_seq = str(row["sequence"])

    seq = "".join(
        aa
        for aa in raw_seq
        if aa in VALID_AA
    )

    seq = seq[:150]

    print(
        f"Running {i+1}/{len(top)}"
    )

    if len(seq) < 5:

        print("Skipped (invalid sequence)")

        structures.append({
            "sequence": raw_seq,
            "status": "invalid"
        })

        continue

    try:

        inputs = tokenizer(
            [seq],
            return_tensors="pt",
            add_special_tokens=False
        )

        if torch.cuda.is_available():

            inputs = {
                k: v.cuda()
                for k, v in inputs.items()
            }

        with torch.no_grad():

            structure_model(**inputs)

        structures.append({
            "sequence": seq,
            "status": "done"
        })

    except Exception as e:

        print("Failed:", str(e))

        structures.append({
            "sequence": seq,
            "status": "failed"
        })

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Structure prediction completed ✔")

# Step Eleven: Export & Final Results

In this step, we export the final ranked mutation results.

The output includes:
- Mutation position
- Wild-type residue
- Mutant residue
- Mutated sequence
- Stability score

This file can be used for:
- downstream bioinformatics analysis
- protein engineering studies
- structure-function interpretation

In [ ]:
import os
import pandas as pd

# validation
if "df" not in globals():

    raise ValueError(
        "DataFrame not found. Run analysis step first."
    )

required_cols = [
    "position",
    "wt",
    "mut",
    "sequence",
    "score"
]

missing = [
    c
    for c in required_cols
    if c not in df.columns
]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )

if len(df) == 0:

    raise ValueError(
        "No mutation results available."
    )

# sort
final_df = (
    df
    .sort_values(
        "score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

# export
output_path = "mutation_results.csv"

final_df.to_csv(
    output_path,
    index=False
)

print("Export completed")
print("Saved:", output_path)

display(
    final_df.head(10)
)

# Final Check: Pipeline Summary

This cell verifies that the pipeline executed successfully and produces a final summary.

In [ ]:
print("\nPipeline executed successfully ✔\n")

if "df" not in globals():

    print(
        "No analysis dataframe found."
    )

else:

    print(
        "Total mutations analyzed:",
        len(df)
    )

    if len(df):

        print(
            "Best score:",
            round(
                df["score"].max(),
                4
            )
        )

        print(
            "Worst score:",
            round(
                df["score"].min(),
                4
            )
        )

        print(
            "Top sequence:"
        )

        print(
            df.iloc[0]["sequence"]
        )

print(
    "\nReady for downstream structure analysis ✔"
)